# Multi-PDF Long-Context Evaluation

Test 3 approaches semantic, keyword, long-context where the entire 3 manuala are fed into the context 

**Models:** GPT-5.4 mini, GPT-5.4 nano  
**QA:** Combined Mill + HAAS Lathe + UR5e Cobot question sets

In [1]:
import sys, json, re
from pathlib import Path
from datetime import date

import pandas as pd
import fitz
import tiktoken
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI
from langchain_community.document_loaders import PyMuPDFLoader

sys.path.insert(0, str(Path("..").resolve())) # because we are pulling these in a jupyter notebook, we need to add the parent directory to the path to access the system module

from system.rag import (
    LLM,
    Approaches,
    RAGExperimentRunner,
    CSVProcessor,
    LangfairMetricsCalculator,
    LangfairRunner,
    ApproachRetrievers,
)
from system.preprocess import PDFPreprocessor
from system.evaluation import (
    JudgeBatchConfig,
    JudgeBatchBuilder,
    BatchResultsConfig,
    BatchResultsExporter,
    JudgeMergeConfig,
    JudgeResultsMerger,
    CSVColumnMergeConfig,
    CSVColumnMerger,
)
from system.utils import EnvironmentConfig, read_text


/Users/ryan/Desktop/Work/SIGHT/safety_rag_eval_ryan/venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


# Run Directory Setup

All artifacts for this run live under one directory.

In [2]:
RUN_ID = f"{date.today().isoformat()}_multi_pdf_eval"
RUN_DIR = Path(f"../data/results/runs/multi_pdf/{RUN_ID}")

for sub in ["rag", "raw", "parsed", "merged", "meta"]:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print(RUN_ID)
print(f"Run directory: {RUN_DIR.resolve()}")


2026-04-12_multi_pdf_eval
Run directory: /Users/ryan/Desktop/Work/SIGHT/safety_rag_eval_ryan/data/results/runs/multi_pdf/2026-04-12_multi_pdf_eval


# Crop PDFs

Crop all 3 manuals to remove headers/footers and reduce token count.

In [3]:
PDF_DIR = Path("../data/input/input_pdfs")
CROP_DIR = Path("../data/input/cropped_pdfs")
CROP_DIR.mkdir(parents=True, exist_ok=True)

TARGETS = {
    "Mill (Bridgeport)": {
        "path": PDF_DIR / "Bridgeport Series 1 Milling manual with schematics.pdf",
        "crop_top": 0.04, "crop_bottom": 0.075, "crop_left": 0.0, "crop_right": 0.0,
    },
    "Lathe (HAAS TL-1)": {
        "path": PDF_DIR / "TL1_lathe operator's-manual.pdf",
        "crop_percent": 0.075,
    },
    "UR5e Cobot": {
        "path": PDF_DIR / "UR5e_Universal_Robots User Manual.pdf",
        "crop_percent": 0.075,
    },
}

CROPPED = {}
for label, cfg in TARGETS.items():
    out = CROP_DIR / f"cropped_{cfg['path'].name}"
    PDFPreprocessor.crop_pdf(
        cfg["path"], out,
        crop_percent=cfg.get("crop_percent", 0.075),
        crop_top=cfg.get("crop_top"),
        crop_bottom=cfg.get("crop_bottom"),
        crop_left=cfg.get("crop_left"),
        crop_right=cfg.get("crop_right"),
    )
    CROPPED[label] = out
    print(f"Cropped {label} -> {out.name}")

Cropped Mill (Bridgeport) -> cropped_Bridgeport Series 1 Milling manual with schematics.pdf
Cropped Lathe (HAAS TL-1) -> cropped_TL1_lathe operator's-manual.pdf
Cropped UR5e Cobot -> cropped_UR5e_Universal_Robots User Manual.pdf


# Load Cropped Text & Token Counts

In [4]:
def load_pdf_as_text(path) -> str:
    pages = PyMuPDFLoader(str(path)).load()
    return "\n\n".join(p.page_content for p in pages).strip()

enc = tiktoken.get_encoding("cl100k_base")
cropped_texts = {}
total_tokens = 0

for label in TARGETS:
    text = load_pdf_as_text(CROPPED[label])
    cropped_texts[label] = text
    n_tokens = len(enc.encode(text))
    total_tokens += n_tokens
    print(f"{label:25s}  {len(text):>10,} chars   {n_tokens:>8,} tokens")

print(f"\n{'COMBINED':25s}  {'':>10s}        {total_tokens:>8,} tokens")

Mill (Bridgeport)             121,502 chars     45,319 tokens
Lathe (HAAS TL-1)             557,235 chars    154,558 tokens
UR5e Cobot                    253,244 chars     60,754 tokens

COMBINED                                      260,631 tokens


In [5]:

print(cropped_texts["UR5e Cobot"][4500:5000])

print("================================\n")
print(cropped_texts["Lathe (HAAS TL-1)"][900:5000])


ccessing Robot Data
204
16. Disposal and Environment
206
17. Declarations and Certificates (original EN)
208
18. Declarations and Certificates
210
19. Safety Functions Table
212
19.1. Table 1a
219
19.2. Table 2
220
20. Certifications
224
21. Certificates
226



1. Liability and Intended Use
1.1. Limitation of Liability
Description
Any information provided in this manual must not be construed as a warranty, by UR,
that the industrial robot will not cause injury or damage, even if the industrial r

lity 
damages resulting from the use of the information contained in this publication

s product uses Java Technology from Oracle Corporation and we request that you acknowledge that 
acle owns the Java Trademark and all Java related Trademarks and agree to comply with the 
demark guidelines at www.oracle.com/us/legal/third-party-trademarks/index.html. 
y further distribution of the Java programs (beyond this appliance/machine) is subject to a legally 
ding End User License Agreement with Orac

# Combine QA Sets

Merge Mill, HAAS Lathe, and UR5e Cobot QA into a single CSV with a `machine` column.

In [6]:
QA_SOURCES = {
    "Mill (Bridgeport)": Path("../data/QA/MILL/Mill Feedback Accepted.csv"),
    "Lathe (HAAS TL-1)": Path("../data/QA/HAAS/Final HAAS Lathe QA.csv"),
    "UR5e Cobot":        Path("../data/QA/COBOT/Final_COBOT_QA.csv"),
}

frames = []
for machine, path in QA_SOURCES.items():
    tmp = pd.read_csv(path)[["question", "gold_answer"]].copy()
    tmp.insert(0, "machine", machine)
    frames.append(tmp)

combined_qa = pd.concat(frames, ignore_index=True)

COMBINED_QA = RUN_DIR / "meta" / "combined_multi_pdf_qa.csv"
combined_qa.to_csv(COMBINED_QA, index=False)

print(f"Total questions: {len(combined_qa)}")
print(combined_qa["machine"].value_counts())

Total questions: 162
machine
UR5e Cobot           60
Mill (Bridgeport)    51
Lathe (HAAS TL-1)    51
Name: count, dtype: int64


In [7]:
## not null in any columns
combined_qa.isnull().sum()


machine        0
question       0
gold_answer    0
dtype: int64

In [8]:
combined_qa.sample(5)

,machine,question,gold_answer
92,Lathe (HAAS TL-1),What setting number prevents the operator from...,Setting #119 or Setting #8\n
7,Mill (Bridgeport),What is the lubrication schedule for the bridg...,"At least weekly, if not daily for the quill."
55,Lathe (HAAS TL-1),What are the typical noise limits of the HAAS ...,Typical A-Weighted sound pressure measurements...
66,Lathe (HAAS TL-1),I am encountering an issue/error with the UI o...,You should email an error report to a local HA...
120,UR5e Cobot,I dropped some liquid on my robot arm. Will th...,It has the potential for a break or safety iss...


# Configure & Run Experiment

In [9]:
env_config = EnvironmentConfig()
rets = ApproachRetrievers(env_config)

# Long-context gets ALL 3 manuals
rets.set_long_context_texts(cropped_texts)

runner = RAGExperimentRunner(
    retrievers=rets,
    num_replicates=1,
    approaches=Approaches.OPENAI_SEMANTIC | Approaches.OPENAI_KEYWORD | Approaches.LONG_CONTEXT,
    models=LLM.GPT_5_4_MINI_2026_03_17 | LLM.GPT_5_4_NANO_2026_03_17,
    max_tokens_list=[5000],
    efforts=["low"],
    topk_list=[3],
    ans_instr_A=read_text("../data/prompts/ans_instr_A.txt"),
    fewshot_A=read_text("../data/prompts/fewshot_A.txt"),
    max_concurrent=5,
    max_chars_per_content=500_000,
    include_hits_text=False,
)
print(runner)

RAGExperimentRunner Configuration
  Approaches       : ['openai_keyword', 'openai_semantic', 'long_context']
  Models           : ['gpt-5.4-mini-2026-03-17', 'gpt-5.4-nano-2026-03-17']
  Max tokens       : [5000]
  Efforts          : ['low']
  Top-k            : [3]
  Answer instr IDs : ['A']
  Few-shot IDs     : ['A']
  Replicates       : 1
  Max concurrent   : 5
  Max chars/content: 500,000
  Include hits text: False
  Min words subsplit: 3000


In [10]:
# test_combined_qa = combined_qa.sample(n=2)
# test_qa_path = RUN_DIR / "meta" / "test_combined_qa.csv"
# test_combined_qa.to_csv(test_qa_path, index=False)

# test_combined_qa.shape

In [11]:
combined_qa.shape

(162, 3)

In [12]:
combined_qa = pd.read_csv(COMBINED_QA)

In [13]:
combined_qa.shape

(162, 3)

In [14]:
RAG_OUTPUT = RUN_DIR / "rag" / "MULTI_PDF_OUTPUT_test.csv"

await runner.run(COMBINED_QA, RAG_OUTPUT)

RAGExperimentRunner Configuration
  Approaches       : ['openai_keyword', 'openai_semantic', 'long_context']
  Models           : ['gpt-5.4-mini-2026-03-17', 'gpt-5.4-nano-2026-03-17']
  Max tokens       : [5000]
  Efforts          : ['low']
  Top-k            : [3]
  Answer instr IDs : ['A']
  Few-shot IDs     : ['A']
  Replicates       : 1
  Max concurrent   : 5
  Max chars/content: 500,000
  Include hits text: False
  Min words subsplit: 3000
----------------------------------------
  Questions        : 162
  Total permutations: 972


AuthenticationError: Error code: 401 - {'error': {'message': "You have insufficient permissions for this operation. Missing scopes: api.responses.write. Check that you have the correct role in your organization (Reader, Writer, Owner) and project (Member, Owner), and if you're using a restricted API key, that it has the necessary scopes.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [12]:
df = pd.read_csv(RAG_OUTPUT)
print(f"Rows: {len(df)}")
print(f"Approaches: {sorted(df['approach'].unique())}")
print(f"Models: {sorted(df['model'].unique()) if 'model' in df.columns else sorted(df['meta_model'].unique())}")
df.tail(3)

Rows: 12
Approaches: ['long_context', 'openai_keyword', 'openai_semantic']
Models: ['gpt-5.4-mini-2026-03-17', 'gpt-5.4-nano-2026-03-17']


,permutation_id,time_started,time_ended,total_elapsed_time,min_words_for_subsplit,approach,model,max_tokens,reasoning_effort,top_k,...,generated_answer,retrieved_files,meta_resp_id,meta_model,meta_status,meta_created,meta_input_tokens,meta_output_tokens,meta_total_tokens,meta_reason
9,eyJtZXRhZGF0YSI6eyJhbnN3ZXJfaW5zdHJ1Y3Rpb25zX2...,2026-04-12 18:46:11 EDT,2026-04-12 18:46:15 EDT,3.76 Seconds,3000,long_context,gpt-5.4-mini-2026-03-17,5000,low,3,...,Thoroughly clean the protective coating from t...,Mill (Bridgeport);Lathe (TL1);UR5e Cobot,resp_05dbb852b4f514ba0069dc20b4754c819f8011218...,gpt-5.4-mini-2026-03-17,completed,1.776034e+09,247745,77,247822,NaN
10,eyJtZXRhZGF0YSI6eyJhbnN3ZXJfaW5zdHJ1Y3Rpb25zX2...,2026-04-12 18:46:16 EDT,2026-04-12 18:46:23 EDT,7.53 Seconds,3000,long_context,gpt-5.4-nano-2026-03-17,5000,low,3,...,"To set tool offsets on the Haas lathe, you fir...",Mill (Bridgeport);Lathe (TL1);UR5e Cobot,resp_04db34526d69e3ae0069dc20b8ec18819d9065b39...,gpt-5.4-nano-2026-03-17,completed,1.776034e+09,247756,212,247968,NaN
11,eyJtZXRhZGF0YSI6eyJhbnN3ZXJfaW5zdHJ1Y3Rpb25zX2...,2026-04-12 18:46:16 EDT,2026-04-12 18:46:24 EDT,8.47 Seconds,3000,long_context,gpt-5.4-nano-2026-03-17,5000,low,3,...,When you first receive the Bridgeport Series I...,Mill (Bridgeport);Lathe (TL1);UR5e Cobot,resp_0cdb29854df7c05e0069dc20b8d064819ea80de59...,gpt-5.4-nano-2026-03-17,completed,1.776034e+09,247745,257,248002,NaN


# Compute Similarity Metrics

In [ ]:
metrics_runner = LangfairRunner(
    calculator=LangfairMetricsCalculator(),
    processor=CSVProcessor(),
    max_concurrent=500,
)
await metrics_runner.run(q_a_csv=RAG_OUTPUT, out_csv=None)

# Judge Batch (Helpfulness & Correctness)

In [ ]:
judge_config = JudgeBatchConfig(
    csv_path=RAG_OUTPUT,
    output_jsonl=RUN_DIR / "raw" / "MULTI_PDF_BATCH.jsonl",
    judge_model="gpt-5",
    completion_window="24h",
    submit_to_openai=True,
    env_file=None,
)
builder = JudgeBatchBuilder(judge_config)
result = builder.run()
print(f"Prepared {result['num_requests']} requests (submitted={result['submitted']})")

In [ ]:
batch_id = {"MULTI_PDF_BATCH": result["submission"]["batch_id"]}

batch_ids_path = RUN_DIR / "meta" / "batch_ids.json"
with batch_ids_path.open("w") as f:
    json.dump(batch_id, f, indent=2)

print(f"Batch ID: {batch_id}")
print(f"Saved to: {batch_ids_path}")

# Check Batch Status

Re-run this cell until status = "completed".

In [ ]:
client = OpenAI()
for label, b_id in batch_id.items():
    batch = client.batches.retrieve(b_id)
    counts = batch.request_counts
    print(f"Batch ID  : {batch.id} ({label})")
    print(f"Status    : {batch.status}")
    print(f"Completed : {counts.completed} / {counts.total}")
    print(f"Failed    : {counts.failed}")
    print()

# Parse Judge Results & Merge

Run after batch completes. Produces the final merged CSV.

In [ ]:
MERGED_OUTPUT = RUN_DIR / "merged" / "multi_pdf_eval.csv"

cfg = BatchResultsConfig(
    batch_id=batch_id["MULTI_PDF_BATCH"],
    raw_jsonl_path=RUN_DIR / "raw" / "multi_pdf_raw.jsonl",
    json_output_path=RUN_DIR / "parsed" / "multi_pdf.json",
    csv_output_path=RUN_DIR / "parsed" / "multi_pdf.csv",
)
BatchResultsExporter().run(cfg)

merge_cfg = JudgeMergeConfig(
    input_csv=cfg.csv_output_path,
    output_csv=RUN_DIR / "parsed" / "multi_pdf_wide.csv",
)
JudgeResultsMerger().run(merge_cfg)

RAG_WITH_METRICS = Path(str(RAG_OUTPUT).replace(".csv", "_with_metrics.csv"))
CSVColumnMerger().run(CSVColumnMergeConfig(
    left_csv=RAG_WITH_METRICS,
    right_csv=merge_cfg.output_csv,
    output_csv=MERGED_OUTPUT,
))
print(f"Final merged CSV: {MERGED_OUTPUT}")
print(f"Rows: {len(pd.read_csv(MERGED_OUTPUT))}")

In [ ]:
manifest = {
    "run_id": RUN_ID,
    "approach": "semantic + keyword + long_context (all 3 PDFs)",
    "datasets": ["Mill (Bridgeport)", "Lathe (HAAS TL-1)", "UR5e Cobot"],
    "models": ["gpt-5.4-mini-2026-03-17", "gpt-5.4-nano-2026-03-17"],
    "created_date": date.today().isoformat(),
    "source_input_csvs": [str(p) for p in QA_SOURCES.values()],
    "notebook": "system/main_multi_pdf_eval.ipynb",
    "canonical_outputs": {
        "rag": str(RAG_OUTPUT),
        "merged": str(MERGED_OUTPUT),
    },
}
manifest_path = RUN_DIR / "meta" / "run_manifest.json"
with manifest_path.open("w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest saved: {manifest_path}")

# Analysis

Helper functions replicated from `analysis.ipynb` / `analysis_cobot_ur5e.ipynb`.

In [ ]:
MINI  = "gpt-5.4-mini-2026-03-17"
NANO  = "gpt-5.4-nano-2026-03-17"

def load(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    for col in ["judge_answer_correctness_vs_ref", "judge_answer_helpfulness"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1", "yes"])
    def _secs(x):
        m = re.search(r"([\d\.]+)", str(x))
        return float(m.group(1)) if m else None
    if "total_elapsed_time" in df.columns:
        df["latency_sec"] = df["total_elapsed_time"].apply(_secs)
    rates = {"nano": 0.40, "mini": 2.00, "5.4": 3.00, "opus": 15.00}
    def _price(row):
        tokens = row.get("meta_total_tokens", 0) or 0
        mdl = str(row.get("model", "")).lower()
        rate = next((v for k, v in rates.items() if k in mdl), 1.00)
        return (tokens / 1_000_000) * rate
    df["price_usd"] = df.apply(_price, axis=1)
    return df

def rank_table(df, group_cols=["approach", "model"]):
    agg = (
        df.groupby(group_cols)
        .agg(
            avg_tokens   =("meta_total_tokens", "mean"),
            avg_price    =("price_usd", "mean"),
            avg_latency  =("latency_sec", "mean"),
            n_correct    =("judge_answer_correctness_vs_ref", "sum"),
            n_helpful    =("judge_answer_helpfulness", "sum"),
            n_total      =("judge_answer_correctness_vs_ref", "count"),
        )
        .reset_index()
    )
    agg["pct_correct"]  = agg["n_correct"]  / agg["n_total"] * 100
    agg["pct_helpful"]  = agg["n_helpful"]  / agg["n_total"] * 100
    agg["avg_score"]    = (agg["pct_correct"] + agg["pct_helpful"]) / 2
    return agg.sort_values("avg_score", ascending=False).reset_index(drop=True)

def bar_charts(agg, title_suffix="", group="approach", hue="model"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, col, label in zip(axes, ["pct_correct","pct_helpful"], ["% Correctness","% Helpfulness"]):
        sns.barplot(data=agg.sort_values(col, ascending=False), x=col, y=group, hue=hue, ax=ax)
        ax.set_title(f"{label} — {title_suffix}")
        ax.set_xlabel(label); ax.set_ylabel(group)
    plt.tight_layout(); plt.show()

def short_q(q, n=65):
    return q[:n] + "..." if len(q) > n else q

print("Analysis helpers loaded.")

## Overall Ranking: Approach x Model

In [ ]:
df = load(MERGED_OUTPUT)
print(f"Rows: {len(df)}, Approaches: {sorted(df['approach'].unique())}, Models: {sorted(df['model'].unique())}")

agg = rank_table(df, ["approach", "model"])
display(agg[["approach","model","avg_latency","avg_price","pct_correct","pct_helpful","avg_score"]])
bar_charts(agg, title_suffix="Multi-PDF — All Approaches (mini + nano)")

## Approach Ranking (collapsed across models)

In [ ]:
agg_approach = rank_table(df, ["approach"])
display(agg_approach[["approach","avg_latency","avg_price","pct_correct","pct_helpful","avg_score"]])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, label in zip(axes, ["pct_correct","pct_helpful"], ["% Correctness","% Helpfulness"]):
    sns.barplot(data=agg_approach, x="approach", y=col, ax=ax)
    ax.set_title(f"{label} — by Approach")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## Question-Level Breakdown

In [ ]:
q_agg = (
    df.groupby("question")
    .agg(
        n=("judge_answer_correctness_vs_ref", "count"),
        pct_correct=("judge_answer_correctness_vs_ref", lambda x: x.mean() * 100),
        pct_helpful=("judge_answer_helpfulness", lambda x: x.mean() * 100),
    )
    .sort_values("pct_correct", ascending=False)
    .reset_index()
)
display(q_agg)

## Heatmap: Question x Approach

In [ ]:
piv = (
    df.groupby(["question", "approach"])
    ["judge_answer_correctness_vs_ref"].mean()
    .unstack("approach") * 100
)
piv.index = piv.index.map(short_q)
piv = piv.loc[piv.mean(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(10, max(18, len(piv) * 0.35)))
sns.heatmap(piv, annot=True, fmt=".0f", cmap="RdYlGn",
            vmin=0, vmax=100, linewidths=0.5, ax=ax)
ax.set_title("% Correct by Question x Approach (pooled across models)")
plt.tight_layout(); plt.show()

## Per-Machine Breakdown

Does the LLM handle some manuals better than others when all 3 are in context?

In [ ]:
# Map questions back to their machine via the combined QA CSV
qa_map = pd.read_csv(COMBINED_QA).set_index("question")["machine"]
df["machine"] = df["question"].map(qa_map)

agg_machine = rank_table(df, ["machine", "approach", "model"])
display(agg_machine[["machine","approach","model","pct_correct","pct_helpful","avg_score"]])

# Bar chart: machine x approach
agg_m = rank_table(df, ["machine", "approach"])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, col, label in zip(axes, ["pct_correct","pct_helpful"], ["% Correctness","% Helpfulness"]):
    sns.barplot(data=agg_m, x="approach", y=col, hue="machine", ax=ax)
    ax.set_title(f"{label} — by Machine x Approach")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## Difficulty Tier Analysis

In [ ]:
q_overall = df.groupby("question")["judge_answer_correctness_vs_ref"].mean() * 100
q_tiers = pd.cut(q_overall, bins=[-1, 0, 50, 99, 100],
                 labels=["0% (never correct)", "1-50%", "51-99%", "100% (always correct)"])
q_tiers.name = "tier"

tier_counts = q_tiers.value_counts().sort_index()
print("Questions per difficulty tier:")
display(tier_counts.to_frame("n_questions"))

# Per-approach performance within each tier
df_tiered = df.merge(q_tiers.reset_index(), on="question")
for tier_name in tier_counts.index:
    subset = df_tiered[df_tiered["tier"] == tier_name]
    if len(subset) == 0:
        continue
    print(f"\n--- {tier_name} ({tier_counts[tier_name]} questions) ---")
    agg_t = rank_table(subset, ["approach"])
    display(agg_t[["approach","pct_correct","pct_helpful","avg_score"]])

## Universally Missed Questions

Questions with 0% correctness across all approaches and models.

In [ ]:
zero_qs = q_overall[q_overall == 0].index.tolist()
print(f"{len(zero_qs)} questions scored 0% across ALL approaches and models:\n")

for i, q in enumerate(zero_qs, 1):
    print(f"{'='*80}")
    print(f"Q{i}: {q}\n")

    gold = df.loc[df["question"] == q, "gold_answer"].iloc[0]
    print(f"GOLD ANSWER:\n{gold}\n")

    sample = df.loc[df["question"] == q].iloc[0]
    print(f"SAMPLE ({sample['approach']}, {sample['model']}):")
    print(f"{str(sample['generated_answer'])[:500]}\n")